# pymfx — Utilities: generate_index · merge · diff

This notebook demonstrates the three utility functions in `pymfx.utils`:

1. **`generate_index(mfx)`** — automatically compute the `[index]` section (bounding box + anomaly count)
2. **`merge(mfx1, mfx2, gap_s)`** — concatenate two flights in temporal order
3. **`diff(mfx1, mfx2)`** — structured comparison between two `.mfx` files

**Requirements:** no extra dependencies — all utilities are pure Python.

In [ ]:
import sys, math, uuid
sys.path.insert(0, '..')  # if running from notebooks/

import pymfx
from pymfx.models import (
    MfxFile, Meta, Trajectory, TrajectoryPoint, SchemaField,
    Events, Event, Index,
)
from pymfx import compute_checksum
from pymfx.utils import generate_index, merge, diff

print(f"pymfx {pymfx.__version__}")

## Helper: build a flight

In [ ]:
def make_mfx(
    drone_id: str = 'DRONE-A',
    n: int = 30,
    base_lat: float = 48.858,
    base_lon: float = 2.295,
    base_speed: float = 6.0,
    base_alt: float = 100.0,
    t_offset: float = 0.0,
    with_events: bool = True,
) -> MfxFile:
    raw_pts = [
        (
            round(t_offset + i * 0.5, 3),
            round(base_lat + i * 0.0001, 7),
            round(base_lon + math.sin(i * 0.3) * 0.0002, 7),
            round(base_alt + math.sin(i * 0.2) * 5, 2),
            round(base_speed + math.cos(i * 0.25) * 1.5, 2),
        )
        for i in range(n)
    ]
    schema_fields = [
        SchemaField('t',        'float', ['no_null']),
        SchemaField('lat',      'float', ['no_null']),
        SchemaField('lon',      'float', ['no_null']),
        SchemaField('alt_m',    'float32'),
        SchemaField('speed_ms', 'float32'),
    ]
    points    = [TrajectoryPoint(t=r[0], lat=r[1], lon=r[2], alt_m=r[3], speed_ms=r[4]) for r in raw_pts]
    raw_lines = [f"{r[0]:.3f} | {r[1]} | {r[2]} | {r[3]} | {r[4]}" for r in raw_pts]

    ev_schema = [
        SchemaField('t',        'float', ['no_null']),
        SchemaField('type',     'str'),
        SchemaField('severity', 'str'),
        SchemaField('detail',   'str'),
    ]
    ev_list = [
        Event(t=t_offset + 1.0, type='takeoff', severity='info',    detail='ok'),
        Event(t=t_offset + 5.0, type='anomaly', severity='warning', detail='wind'),
    ] if with_events else []
    ev_raw = [f"{e.t:.3f} | {e.type} | {e.severity} | {e.detail}" for e in ev_list]

    lats = [p.lat for p in points];  lons = [p.lon for p in points]

    return MfxFile(
        version='1.0', encoding='UTF-8',
        meta=Meta(
            id=f'uuid:{uuid.uuid4()}',
            drone_id=drone_id,
            drone_type='multirotor',
            pilot_id='FR-PILOT-001',
            date_start='2026-06-01T10:00:00Z',
            status='complete',
            application='survey',
            location='Paris, France',
            sensors=['GPS'],
            data_level='raw',
            license='CC-BY-4.0',
            contact='test@example.com',
        ),
        trajectory=Trajectory(
            frequency_hz=2.0,
            schema_fields=schema_fields,
            points=points,
            checksum=compute_checksum(raw_lines),
            raw_lines=raw_lines,
        ),
        events=Events(
            schema_fields=ev_schema,
            events=ev_list,
            checksum=compute_checksum(ev_raw),
            raw_lines=ev_raw,
        ) if ev_list else None,
    )

mfx_a = make_mfx('DRONE-ALPHA', n=30, base_speed=7.0)
mfx_b = make_mfx('DRONE-BETA',  n=20, base_speed=5.5, base_lat=48.862)
print(f"Flight A: {len(mfx_a.trajectory.points)} pts")
print(f"Flight B: {len(mfx_b.trajectory.points)} pts")

## 1. generate_index

`generate_index(mfx)` automatically computes the `[index]` section:

- **`bbox`** — `(lon_min, lat_min, lon_max, lat_max)` bounding box of all trajectory points
- **`anomalies`** — count of events with `severity == 'warning'` or `'critical'`

This is useful when building a `.mfx` file programmatically or when updating an existing file after editing events.

In [ ]:
idx = generate_index(mfx_a)
print("bbox      :", idx.bbox)
print("anomalies :", idx.anomalies)  # 1 warning event

In [ ]:
# Verify the bbox actually contains all points
lon_min, lat_min, lon_max, lat_max = idx.bbox
for p in mfx_a.trajectory.points:
    assert lon_min <= p.lon <= lon_max, f"lon out of bbox: {p.lon}"
    assert lat_min <= p.lat <= lat_max, f"lat out of bbox: {p.lat}"
print("✓ All trajectory points are inside the bounding box")

In [ ]:
# Typical usage: assign back to mfx.index, then write
mfx_a.index = generate_index(mfx_a)
pymfx.write(mfx_a, '/tmp/flight_a_indexed.mfx')

# Validate (V17 checks bbox, V19 checks anomalies count)
result = pymfx.validate(mfx_a)
print(result)

In [ ]:
# Edge case: no events → anomalies = 0
mfx_noev = make_mfx('NO-EVENTS', with_events=False)
idx_noev = generate_index(mfx_noev)
print("anomalies (no events):", idx_noev.anomalies)  # 0

# Edge case: empty trajectory → bbox = None
mfx_empty = make_mfx('EMPTY')
mfx_empty.trajectory.points = []
idx_empty = generate_index(mfx_empty)
print("bbox (empty traj)    :", idx_empty.bbox)       # None

## 2. merge

`merge(mfx1, mfx2, gap_s=0.0)` concatenates two flights:

- **Trajectory** — all points of `mfx1` followed by `mfx2` (with `t` values of `mfx2` shifted so the time axis stays strictly increasing)
- **Events** — combined from both, with `mfx2` events also time-shifted
- **Schema** — union of both schemas
- **Meta** — copied from `mfx1`, new UUID assigned

In [ ]:
leg1 = make_mfx('LEG-1', n=20)
leg2 = make_mfx('LEG-2', n=15)

combined = merge(leg1, leg2, gap_s=5.0)

print(f"Leg 1  : {len(leg1.trajectory.points)} pts, t = {leg1.trajectory.points[0].t:.1f} … {leg1.trajectory.points[-1].t:.1f} s")
print(f"Leg 2  : {len(leg2.trajectory.points)} pts, t = {leg2.trajectory.points[0].t:.1f} … {leg2.trajectory.points[-1].t:.1f} s")
print(f"Merged : {len(combined.trajectory.points)} pts, t = {combined.trajectory.points[0].t:.1f} … {combined.trajectory.points[-1].t:.1f} s")
print(f"Gap    : 5.0 s — first point of leg2 in merged = {combined.trajectory.points[20].t:.1f} s")

In [ ]:
# t axis must be strictly increasing
ts = [p.t for p in combined.trajectory.points]
assert all(ts[i+1] > ts[i] for i in range(len(ts)-1)), "t not strictly increasing!"
print("✓ t axis strictly increasing across the merged flight")

In [ ]:
# Events are also time-shifted
print("Leg 1 events:",   [(e.t, e.type) for e in leg1.events.events])
print("Leg 2 events:",   [(e.t, e.type) for e in leg2.events.events])
print("Merged events:",  [(e.t, e.type) for e in combined.events.events])

In [ ]:
# Meta comes from mfx1; new UUID is assigned
print("drone_id   :", combined.meta.drone_id)   # 'LEG-1'
print("new UUID   :", combined.meta.id)
print("different? :", combined.meta.id != leg1.meta.id)

In [ ]:
# Write and validate the merged flight
combined.index = generate_index(combined)
pymfx.write(combined, '/tmp/merged_flight.mfx')
merged_back = pymfx.parse('/tmp/merged_flight.mfx')
print(pymfx.validate(merged_back))
print(f"Round-trip: {len(merged_back.trajectory.points)} pts ✓")

In [ ]:
# Also accessible via the public API
combined2 = pymfx.merge(leg1, leg2, gap_s=5.0)
assert len(combined2.trajectory.points) == len(combined.trajectory.points)
print("pymfx.merge() public API ✓")

## 3. diff

`diff(mfx1, mfx2)` returns a `DiffResult` with:

- `meta_diffs` — list of `(field, value_in_mfx1, value_in_mfx2)` for every differing meta field
- Trajectory metrics: `point_count`, `duration_s`, `total_distance_m`, `frequency_hz`
- Event count
- `has_differences` — `True` if anything differs

In [ ]:
result = diff(mfx_a, mfx_b)
print(result)

In [ ]:
print("has_differences:", result.has_differences)
print("meta_diffs     :", result.meta_diffs)
print("point_count    :", result.point_count_1, "vs", result.point_count_2)
print("duration_s     :", result.duration_s_1,  "vs", result.duration_s_2)
print("total_dist_m   :", round(result.total_distance_m_1, 1), "vs", round(result.total_distance_m_2, 1))
print("event_count    :", result.event_count_1, "vs", result.event_count_2)

In [ ]:
# Identical flights → no differences
mfx_x = make_mfx('SAME-DRONE', n=10)
mfx_y = make_mfx('SAME-DRONE', n=10)   # different UUID but same everything else

result_same = diff(mfx_x, mfx_y)
print(result_same)
print("has_differences:", result_same.has_differences)

In [ ]:
# Use via the public API
result2 = pymfx.diff(mfx_a, mfx_b)
assert isinstance(result2, pymfx.DiffResult)
print("pymfx.diff() public API ✓")

In [ ]:
# Practical use: compare before/after edit
import copy
mfx_before = make_mfx('ORIGINAL', n=25, base_speed=8.0)
mfx_after  = copy.deepcopy(mfx_before)
mfx_after.meta.drone_id = 'CORRECTED'
# Trim first 5 points
mfx_after.trajectory.points = mfx_after.trajectory.points[5:]

r = diff(mfx_before, mfx_after)
print(r)
if r.has_differences:
    print(f"→ {len(r.meta_diffs)} meta field(s) changed, point count Δ = {r.point_count_1 - r.point_count_2}")

## Summary

| Function | Returns | Use case |
|---|---|---|
| `pymfx.generate_index(mfx)` | `Index` | Auto-fill `[index]` section before writing |
| `pymfx.merge(mfx1, mfx2, gap_s)` | `MfxFile` | Concatenate multi-leg flights |
| `pymfx.diff(mfx1, mfx2)` | `DiffResult` | Compare two flights for QA / regression |